<a href="https://colab.research.google.com/github/HaiderZahoorr/COMPUTER-VISION/blob/main/Skin-Lesion-Classification-and-Model-Benchmarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Skin Cancer Classification and Model Benchmarking
# -------------------------------------------------
# This notebook/script:
# 1. Downloads the ISIC skin-cancer dataset from Kaggle.
# 2. Finds the Train/Test folders automatically.
# 3. Fine-tunes eight CNN architectures.
# 4. Evaluates them using Accuracy, Precision, Recall, F1 and AUC.
# 5. Uses the strongest CNN as a deep-feature extractor.
# 6. Tests seven classical classifiers on those features.
# 7. Profiles CNN size, FLOPs and inference speed.
# 8. Writes all comparison tables to Excel.

import os
import sys
import time
import subprocess
import importlib.util
from pathlib import Path

# --------------------------- Package Setup ---------------------------

REQUIRED = {
    "kagglehub": "kagglehub",
    "thop": "thop",
    "xgboost": "xgboost",
    "openpyxl": "openpyxl",
}

for module_name, package_name in REQUIRED.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from thop import profile

# --------------------------- Configuration ---------------------------

SEED = 42
DATASET_ID = "nodoubttome/skin-cancer9-classesisic"

IMAGE_DIM = 224
BATCH = 32
TRAIN_EPOCHS = 5
LR = 1e-4
WORKERS = 2

OUTPUT_DIR = Path("results")
WEIGHTS_DIR = OUTPUT_DIR / "weights"
OUTPUT_DIR.mkdir(exist_ok=True)
WEIGHTS_DIR.mkdir(exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# --------------------------- Dataset Discovery ---------------------------

def contains_images(folder):
    valid = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return any(p.is_file() and p.suffix.lower() in valid for p in Path(folder).iterdir())

def locate_split(root, wanted_name):
    root = Path(root)

    for current, dirs, _ in os.walk(root):
        current_path = Path(current)

        if current_path.name.lower() != wanted_name.lower():
            continue

        class_dirs = []
        for child in current_path.iterdir():
            if child.is_dir() and contains_images(child):
                class_dirs.append(child)

        if len(class_dirs) >= 2:
            return current_path

    return None

print("\nDownloading dataset...")
dataset_location = Path(__import__("kagglehub").dataset_download(DATASET_ID))
print("Dataset location:", dataset_location)

TRAIN_DIR = locate_split(dataset_location, "Train")
TEST_DIR = locate_split(dataset_location, "Test")

if TRAIN_DIR is None:
    raise FileNotFoundError("Training folder could not be located.")

print("Train folder:", TRAIN_DIR)
print("Test folder :", TEST_DIR if TEST_DIR else "Not found")

# --------------------------- Image Transforms ---------------------------

train_tfms = transforms.Compose([
    transforms.Resize((IMAGE_DIM, IMAGE_DIM)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMAGE_DIM, IMAGE_DIM)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

base_train = datasets.ImageFolder(TRAIN_DIR)
class_names = base_train.classes
num_classes = len(class_names)

print("\nClasses:", class_names)
print("Number of classes:", num_classes)

# --------------------------- Dataset Wrapper ---------------------------

class TransformedSubset(Dataset):
    def __init__(self, image_folder, indices, transform):
        self.folder = image_folder
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        real_index = self.indices[index]
        image_path, label = self.folder.samples[real_index]
        image = self.folder.loader(image_path)

        if self.transform:
            image = self.transform(image)

        return image, label

all_labels = np.array(base_train.targets)
all_ids = np.arange(len(base_train))

train_ids, validation_ids = train_test_split(
    all_ids,
    test_size=0.15,
    random_state=SEED,
    stratify=all_labels
)

train_set = TransformedSubset(base_train, train_ids, train_tfms)
val_set = TransformedSubset(base_train, validation_ids, eval_tfms)

train_loader = DataLoader(
    train_set, batch_size=BATCH, shuffle=True,
    num_workers=WORKERS, pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_set, batch_size=BATCH, shuffle=False,
    num_workers=WORKERS, pin_memory=torch.cuda.is_available()
)

if TEST_DIR is not None:
    test_base = datasets.ImageFolder(TEST_DIR, transform=eval_tfms)
    test_loader = DataLoader(
        test_base, batch_size=BATCH, shuffle=False,
        num_workers=WORKERS, pin_memory=torch.cuda.is_available()
    )
    evaluation_loader = test_loader
else:
    evaluation_loader = val_loader

# --------------------------- Model Factory ---------------------------

def build_network(name):
    builders = {
        "AlexNet": lambda: models.alexnet(weights=models.AlexNet_Weights.DEFAULT),
        "VGG16": lambda: models.vgg16(weights=models.VGG16_Weights.DEFAULT),
        "VGG19": lambda: models.vgg19(weights=models.VGG19_Weights.DEFAULT),
        "ResNet18": lambda: models.resnet18(weights=models.ResNet18_Weights.DEFAULT),
        "ResNet50": lambda: models.resnet50(weights=models.ResNet50_Weights.DEFAULT),
        "ResNet101": lambda: models.resnet101(weights=models.ResNet101_Weights.DEFAULT),
        "DenseNet121": lambda: models.densenet121(weights=models.DenseNet121_Weights.DEFAULT),
        "EfficientNet-B0": lambda: models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        ),
    }

    net = builders[name]()

    if name.startswith("VGG") or name == "AlexNet":
        net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, num_classes)
    elif name.startswith("ResNet"):
        net.fc = nn.Linear(net.fc.in_features, num_classes)
    elif name == "DenseNet121":
        net.classifier = nn.Linear(net.classifier.in_features, num_classes)
    elif name == "EfficientNet-B0":
        net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, num_classes)

    return net.to(DEVICE)

cnn_names = [
    "AlexNet",
    "VGG16",
    "VGG19",
    "ResNet18",
    "ResNet50",
    "ResNet101",
    "DenseNet121",
    "EfficientNet-B0",
]

# --------------------------- Evaluation ---------------------------

def collect_predictions(net, loader):
    net.eval()
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)

            logits = net(images)
            probabilities = torch.softmax(logits, dim=1)

            y_true.extend(labels.numpy())
            y_pred.extend(probabilities.argmax(1).cpu().numpy())
            y_prob.extend(probabilities.cpu().numpy())

    return np.asarray(y_true), np.asarray(y_pred), np.asarray(y_prob)

def metric_row(y_true, y_pred, probabilities):
    result = {
        "Accuracy (%)": accuracy_score(y_true, y_pred) * 100,
        "Precision (%)": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ) * 100,
        "Recall (%)": recall_score(
            y_true, y_pred, average="weighted", zero_division=0
        ) * 100,
        "F1-Score (%)": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ) * 100,
    }

    try:
        result["AUC (%)"] = roc_auc_score(
            y_true,
            probabilities,
            multi_class="ovr",
            average="weighted"
        ) * 100
    except ValueError:
        result["AUC (%)"] = np.nan

    return result

# --------------------------- Table 1 ---------------------------

table1 = []
trained_models = {}

for name in cnn_names:
    print(f"\nTraining {name}...")
    network = build_network(name)

    optimizer = torch.optim.Adam(network.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(TRAIN_EPOCHS):
        network.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()
            output = network(images)
            loss = loss_fn(output, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(
            f"Epoch {epoch + 1}/{TRAIN_EPOCHS} "
            f"- loss: {running_loss / max(1, len(train_loader)):.4f}"
        )

    y_true, y_pred, probs = collect_predictions(network, evaluation_loader)
    scores = metric_row(y_true, y_pred, probs)
    scores["Model"] = name
    table1.append(scores)

    torch.save(network.state_dict(), WEIGHTS_DIR / f"{name.replace('-', '_')}.pth")
    trained_models[name] = network

    del optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

table1_df = pd.DataFrame(table1)[
    ["Model", "Accuracy (%)", "Precision (%)", "Recall (%)", "F1-Score (%)", "AUC (%)"]
]

best_model_name = table1_df.loc[
    table1_df["Accuracy (%)"].idxmax(), "Model"
]

print("\nBest CNN:", best_model_name)

# --------------------------- Feature Extraction ---------------------------

def feature_extractor(net, name):
    net.eval()

    if name == "AlexNet":
        return nn.Sequential(
            net.features,
            net.avgpool,
            nn.Flatten(),
            *list(net.classifier.children())[:-1]
        )

    if name.startswith("VGG"):
        return nn.Sequential(
            net.features,
            net.avgpool,
            nn.Flatten(),
            *list(net.classifier.children())[:-1]
        )

    if name.startswith("ResNet"):
        return nn.Sequential(*list(net.children())[:-1], nn.Flatten())

    if name == "DenseNet121":
        return nn.Sequential(
            net.features,
            nn.ReLU(inplace=False),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

    if name == "EfficientNet-B0":
        return nn.Sequential(
            net.features,
            net.avgpool,
            nn.Flatten()
        )

    raise ValueError("Unknown architecture")

best_net = trained_models[best_model_name]
feature_net = feature_extractor(best_net, best_model_name).to(DEVICE)

def extract_features(loader):
    feature_net.eval()
    features, labels = [], []

    with torch.no_grad():
        for images, y in loader:
            z = feature_net(images.to(DEVICE))
            features.append(z.cpu().numpy())
            labels.append(y.numpy())

    return np.vstack(features), np.concatenate(labels)

train_features, train_targets = extract_features(train_loader)
eval_features, eval_targets = extract_features(evaluation_loader)

scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features)
eval_features_scaled = scaler.transform(eval_features)

# --------------------------- Table 2 ---------------------------

classifiers = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=SEED
    ),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=SEED, n_jobs=-1
    ),
    "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Linear SVM": SVC(
        kernel="linear", probability=True, random_state=SEED
    ),
    "RBF-SVM": SVC(
        kernel="rbf", probability=True, random_state=SEED
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=num_classes,
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    ),
}

table2 = []

for clf_name, clf in classifiers.items():
    print(f"\nClassifier: {clf_name}")

    clf.fit(train_features_scaled, train_targets)

    predictions = clf.predict(eval_features_scaled)

    if hasattr(clf, "predict_proba"):
        probabilities = clf.predict_proba(eval_features_scaled)
    else:
        decision = clf.decision_function(eval_features_scaled)
        probabilities = torch.softmax(
            torch.tensor(decision, dtype=torch.float32), dim=1
        ).numpy()

    scores = metric_row(eval_targets, predictions, probabilities)
    scores["Feature Extractor"] = "Deep Features"
    scores["Classifier"] = clf_name
    table2.append(scores)

table2_df = pd.DataFrame(table2)[
    [
        "Feature Extractor",
        "Classifier",
        "Accuracy (%)",
        "Precision (%)",
        "Recall (%)",
        "F1-Score (%)",
        "AUC (%)",
    ]
]

# --------------------------- Table 3 ---------------------------

profile_names = [
    "AlexNet",
    "VGG16",
    "VGG19",
    "ResNet18",
    "ResNet50",
    "DenseNet121",
    "EfficientNet-B0",
]

accuracy_lookup = dict(zip(table1_df["Model"], table1_df["Accuracy (%)"]))
table3 = []

dummy = torch.randn(1, 3, IMAGE_DIM, IMAGE_DIM).to(DEVICE)

for name in profile_names:
    net = trained_models[name]
    net.eval()

    parameter_count = sum(p.numel() for p in net.parameters())
    parameter_mb = sum(
        p.numel() * p.element_size() for p in net.parameters()
    ) / (1024 ** 2)

    with torch.no_grad():
        flop_count, _ = profile(net, inputs=(dummy,), verbose=False)

    warmup = 5
    repeats = 20

    with torch.no_grad():
        for _ in range(warmup):
            _ = net(dummy)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        for _ in range(repeats):
            _ = net(dummy)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

    inference_ms = ((time.perf_counter() - start) / repeats) * 1000

    table3.append({
        "Model": name,
        "Parameters (M)": parameter_count / 1e6,
        "Model Size (MB)": parameter_mb,
        "FLOPs (G)": flop_count / 1e9,
        "Inference Time (ms)": inference_ms,
        "Accuracy (%)": accuracy_lookup[name],
    })

table3_df = pd.DataFrame(table3)

# --------------------------- Save Results ---------------------------

excel_file = OUTPUT_DIR / "model_comparison_results.xlsx"

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    table1_df.to_excel(writer, sheet_name="Table 1", index=False)
    table2_df.to_excel(writer, sheet_name="Table 2", index=False)
    table3_df.to_excel(writer, sheet_name="Table 3", index=False)

table2_df.to_csv(OUTPUT_DIR / "classifier_comparison.csv", index=False)

print("\n" + "=" * 70)
print("TABLE 1")
print(table1_df.to_string(index=False))

print("\nTABLE 2")
print(table2_df.to_string(index=False))

print("\nTABLE 3")
print(table3_df.to_string(index=False))

print("\nResults saved to:", excel_file)
print("Finished.")
